In [ ]:
import os
os.chdir('../../../../..')

In [ ]:
import numpy as np

from sklearn.cluster import AgglomerativeClustering, SpectralClustering, DBSCAN, KMeans
from kmedoids import KMedoids

from src.datasets import QM9Dataset
from src.helper_functions import plot_molecules_with_py3dmol, create_chemiscope_viewer, plot_distance_matrix_projection, evaluate_distance_matrix_clustering_sweep, average_numeric_by_cluster

In [ ]:
qm9 = QM9Dataset(limit=5000, sampling_strategy="stratified", stratify_by=["num_atoms", "gap"], add_selfies_transformer=True)
molecules = qm9.get_molecules()
df = qm9.load()


In [ ]:
len(molecules[0:2])

In [ ]:
plot_molecules_with_py3dmol(molecules[0:3])

In [ ]:
dist_matrix = qm9.get_distance_matrix(
    descriptor="transformer",
    dist_type="cosine",
)


# Determining the best number of clusters for each clustering method

In [ ]:
out = evaluate_distance_matrix_clustering_sweep(
    dist_matrix=dist_matrix,
    fingerprint="transformer",
    distance_metric="cosine",
    dataset_name="qm9",
)

In [ ]:
# find the n molecules that are not on the diagonal with the smallest distance
n = 10
# Get the indices of the upper triangle (excluding diagonal)
triu_indices = np.triu_indices_from(dist_matrix, k=1)
# Get the distances and corresponding molecule pairs
distances = dist_matrix[triu_indices]
molecule_pairs = list(zip(triu_indices[0], triu_indices[1]))
# Get the indices of the n smallest distances
smallest_indices = np.argsort(distances)[:n]
# Get the corresponding molecule pairs for the n smallest distances
closest_pairs = [molecule_pairs[i] for i in smallest_indices]
print("Closest molecule pairs (indices):", closest_pairs)
mols = [(molecules[idx1], molecules[idx2]) for idx1, idx2 in closest_pairs]

In [ ]:
print(mols[0])

In [ ]:
plot_molecules_with_py3dmol(mols[1])

# Hiercical Clustering on Distance Matrix

In [ ]:
model_hier = AgglomerativeClustering(metric='precomputed', n_clusters=7, linkage='complete')
labels_hier = model_hier.fit_predict(dist_matrix)
df = df.with_columns(labels_hier=labels_hier)

In [ ]:
create_chemiscope_viewer(df, dist_matrix, labels_hier, 'PCA')

In [ ]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="transformer",
    distance_metric="cosine",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_hier,
    clustering_method="hierarchical"
)

In [ ]:
average_numeric_by_cluster(df, "labels_hier")

# KMedoids

In [ ]:
model_km = KMedoids(n_clusters=2, metric="precomputed")
labels_km = model_km.fit_predict(dist_matrix)
df = df.with_columns(labels_km=labels_km)

In [ ]:
create_chemiscope_viewer(df, dist_matrix, labels_km, 'PCA')

In [ ]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="transformer",
    distance_metric="cosine",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_km,
    clustering_method="kmedoids"
)

In [ ]:
average_numeric_by_cluster(df, "labels_km")

# Spectral

In [ ]:
model_spectral = SpectralClustering(
                n_clusters=6,
                affinity="precomputed",
                assign_labels='kmeans',
                random_state=42,
            )

labels_spectral = model_spectral.fit_predict(dist_matrix)
df = df.with_columns(labels_spectral=labels_spectral)

In [ ]:
create_chemiscope_viewer(df, dist_matrix, labels_spectral, 'PCA')

In [ ]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="transformer",
    distance_metric="cosine",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_spectral,
    clustering_method="spectral"
)

In [ ]:
average_numeric_by_cluster(df, "labels_spectral")

# DBSCAN 

In [ ]:
model_db = DBSCAN(
    eps=0.5,
    min_samples=2,
    metric='precomputed',
)

labels_db = model_db.fit_predict(dist_matrix)
df = df.with_columns(labels_db=labels_db)

In [ ]:
create_chemiscope_viewer(df, dist_matrix, labels_db, 'PCA')

In [ ]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="transformer",
    distance_metric="cosine",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_db,
    clustering_method="dbscan"
)

In [ ]:
average_numeric_by_cluster(df, "labels_db")

# KMeans on Raw Embeddings


In [ ]:
X_raw = np.array(df["selfies_transformer"].to_list(), dtype=np.float32)
kmeans_raw = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_kmeans_raw = kmeans_raw.fit_predict(X_raw)
df = df.with_columns(labels_kmeans_raw=labels_kmeans_raw)


In [ ]:
average_numeric_by_cluster(df, "labels_kmeans_raw")
